# Packages

In [42]:
import pandas as pd
import numpy as np
import os
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
dir = os.getenv('DIR')
os.getcwd()
os.chdir(dir)
path = os.getcwd()

In [3]:
df = pd.read_csv(os.path.join(path,'data','final','Dataset.csv'))

In [4]:
df

,Key,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,DR,...,Blk,PF,FG%,3P%,FT%,Possessions,NumOT,GamesPlayed,WinRate,RollingScore
0,2003-1102,0.883,0.653,0.806,0.473,0.670,0.543,0.629,0.358,0.626,...,0.223,0.648,0.087,0.068,0.108,0.873,0.000,0.732,0.429,0.876
1,2003-1103,0.951,0.725,0.878,0.405,0.617,0.651,0.715,0.517,0.661,...,0.262,0.660,0.087,0.063,0.120,0.926,0.057,0.724,0.482,0.939
2,2003-1104,0.923,0.699,0.882,0.432,0.657,0.600,0.669,0.580,0.698,...,0.344,0.641,0.077,0.062,0.117,0.913,0.008,0.739,0.587,0.926
3,2003-1105,0.931,0.703,0.899,0.467,0.669,0.608,0.680,0.581,0.692,...,0.245,0.664,0.073,0.067,0.117,0.945,0.032,0.716,0.270,0.947
4,2003-1106,0.906,0.694,0.876,0.426,0.636,0.534,0.622,0.562,0.698,...,0.309,0.642,0.077,0.066,0.106,0.918,0.008,0.732,0.465,0.903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14306,2026-3477,0.901,0.684,0.889,0.427,0.664,0.549,0.612,0.474,0.687,...,0.454,0.610,0.071,0.060,0.121,0.926,0.000,0.716,0.424,0.896
14307,2026-3478,0.877,0.654,0.862,0.432,0.666,0.538,0.587,0.474,0.678,...,0.401,0.609,0.069,0.060,0.124,0.907,0.015,0.739,0.345,0.883
14308,2026-3479,0.908,0.679,0.880,0.510,0.733,0.546,0.603,0.453,0.622,...,0.313,0.638,0.071,0.063,0.122,0.927,0.008,0.732,0.465,0.933
14309,2026-3480,0.929,0.701,0.884,0.479,0.705,0.595,0.668,0.500,0.694,...,0.443,0.627,0.077,0.062,0.116,0.934,0.037,0.724,0.556,0.929


In [20]:
batch_size = 64
teams = 2
d_input = df.shape[1]-1


# Model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

In [19]:
train_df = df[df['Key'].apply(lambda x: int(x.split('-')[0])) <= 2019]
val_df   = df[(df['Key'].apply(lambda x: int(x.split('-')[0])) > 2019) & (df['Key'].apply(lambda x: int(x.split('-')[0])) <= 2022)]
test_df  = df[df['Key'].apply(lambda x: int(x.split('-')[0])) > 2022]

## Components

## Training only the Encoder first

In [11]:
df

,Key,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,DR,...,Blk,PF,FG%,3P%,FT%,Possessions,NumOT,GamesPlayed,WinRate,RollingScore
0,2003-1102,0.883,0.653,0.806,0.473,0.670,0.543,0.629,0.358,0.626,...,0.223,0.648,0.087,0.068,0.108,0.873,0.000,0.732,0.429,0.876
1,2003-1103,0.951,0.725,0.878,0.405,0.617,0.651,0.715,0.517,0.661,...,0.262,0.660,0.087,0.063,0.120,0.926,0.057,0.724,0.482,0.939
2,2003-1104,0.923,0.699,0.882,0.432,0.657,0.600,0.669,0.580,0.698,...,0.344,0.641,0.077,0.062,0.117,0.913,0.008,0.739,0.587,0.926
3,2003-1105,0.931,0.703,0.899,0.467,0.669,0.608,0.680,0.581,0.692,...,0.245,0.664,0.073,0.067,0.117,0.945,0.032,0.716,0.270,0.947
4,2003-1106,0.906,0.694,0.876,0.426,0.636,0.534,0.622,0.562,0.698,...,0.309,0.642,0.077,0.066,0.106,0.918,0.008,0.732,0.465,0.903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14306,2026-3477,0.901,0.684,0.889,0.427,0.664,0.549,0.612,0.474,0.687,...,0.454,0.610,0.071,0.060,0.121,0.926,0.000,0.716,0.424,0.896
14307,2026-3478,0.877,0.654,0.862,0.432,0.666,0.538,0.587,0.474,0.678,...,0.401,0.609,0.069,0.060,0.124,0.907,0.015,0.739,0.345,0.883
14308,2026-3479,0.908,0.679,0.880,0.510,0.733,0.546,0.603,0.453,0.622,...,0.313,0.638,0.071,0.063,0.122,0.927,0.008,0.732,0.465,0.933
14309,2026-3480,0.929,0.701,0.884,0.479,0.705,0.595,0.668,0.500,0.694,...,0.443,0.627,0.077,0.062,0.116,0.934,0.037,0.724,0.556,0.929


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14311 entries, 2003-1102 to 2026-3481
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Score         14311 non-null  float64
 1   FGM           14311 non-null  float64
 2   FGA           14311 non-null  float64
 3   FGM3          14311 non-null  float64
 4   FGA3          14311 non-null  float64
 5   FTM           14311 non-null  float64
 6   FTA           14311 non-null  float64
 7   OR            14311 non-null  float64
 8   DR            14311 non-null  float64
 9   Ast           14311 non-null  float64
 10  TO            14311 non-null  float64
 11  Stl           14311 non-null  float64
 12  Blk           14311 non-null  float64
 13  PF            14311 non-null  float64
 14  FG%           14311 non-null  float64
 15  3P%           14311 non-null  float64
 16  FT%           14311 non-null  float64
 17  Possessions   14311 non-null  float64
 18  NumOT         14311

In [5]:
keys = df.index.tolist()   # Storing the keys
# keys

In [6]:
df.set_index('Key',inplace=True)

In [7]:
class TeamDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data.values, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [8]:
class TeamEncoder(nn.Module):
    def __init__(self, input_dim, d_model=128, n_heads=4, n_layers=2):
        super().__init__()

        # Project stats → embedding
        self.input_proj = nn.Linear(input_dim, d_model)

        # CLS token
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Decoder (for reconstruction)
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        # x: (B, input_dim)

        B = x.size(0)

        x = self.input_proj(x)              # (B, d_model)
        x = x.unsqueeze(1)                 # (B, 1, d_model)

        cls = self.cls.expand(B, 1, -1)    # (B, 1, d_model)

        x = torch.cat([cls, x], dim=1)     # (B, 2, d_model)

        x = self.encoder(x)                # (B, 2, d_model)

        cls_out = x[:, 0, :]               # (B, d_model)

        recon = self.decoder(cls_out)      # (B, input_dim)

        return recon, cls_out

In [39]:
def train_encoder(model, dataloader, epochs=20, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        total_loss = 0

        for batch in dataloader:
            x = batch.to(device)  # (B, input_dim)

            recon, _ = model(x)

            loss = criterion(recon, x)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [41]:
teams = TeamDataset(df)
teamsdataloader = torch.utils.data.DataLoader(teams, batch_size=64, shuffle=False)

encoder = TeamEncoder(input_dim=df.shape[1])

train_encoder(encoder, teamsdataloader)

Epoch 1, Loss: 1.4962
Epoch 2, Loss: 0.3321
Epoch 3, Loss: 0.2621
Epoch 4, Loss: 0.2291
Epoch 5, Loss: 0.1842
Epoch 6, Loss: 0.1470
Epoch 7, Loss: 0.1083
Epoch 8, Loss: 0.1008
Epoch 9, Loss: 0.0812
Epoch 10, Loss: 0.0754
Epoch 11, Loss: 0.0736
Epoch 12, Loss: 0.0541
Epoch 13, Loss: 0.0530
Epoch 14, Loss: 0.0629
Epoch 15, Loss: 0.0605
Epoch 16, Loss: 0.0663
Epoch 17, Loss: 0.0610
Epoch 18, Loss: 0.0457
Epoch 19, Loss: 0.0482
Epoch 20, Loss: 0.0499


In [ ]:
class MatchPredictor(nn.Module):
    def __init__(self, encoder, d_model):
        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, teamA, teamB):
        _, zA = self.encoder(teamA)   # (B, d)
        _, zB = self.encoder(teamB)

        x = torch.cat([
            zA - zB,
            zA * zB,
            zA,
            zB
        ], dim=-1)

        return torch.sigmoid(self.classifier(x))

## Match pairs making

In [ ]:
matchups = pd.read_csv(os.path.join(path,'data','final','Matchups.csv'))


In [12]:
target = matchups[['Key','Win']]

In [68]:
target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 428892 entries, 0 to 428891
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   Key     428892 non-null  object
 1   Win     428892 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 6.5+ MB


In [13]:
target['TeamAID'] = target['Key'].apply(lambda x: x.split('-')[0] + '-' + x.split('-')[1])
target['TeamBID'] = target['Key'].apply(lambda x: x.split('-')[0] + '-' + x.split('-')[2])

In [14]:
target.index

RangeIndex(start=0, stop=428892, step=1)

### Approach 2

In [43]:
encoder.eval()

team_embeddings = {}

with torch.no_grad():
    for team_id, stats in df.iterrows():
        stats = torch.tensor(stats.values, dtype=torch.float32).unsqueeze(0).to(device)

        _, emb = encoder(stats)   # (1, d)

        team_embeddings[team_id] = emb.squeeze(0).cpu()

In [44]:
team_ids = sorted(team_embeddings.keys())

team_embeddings = torch.stack([
    team_embeddings[tid] for tid in team_ids
])

In [45]:
team_ids

['2003-1102',
 '2003-1103',
 '2003-1104',
 '2003-1105',
 '2003-1106',
 '2003-1107',
 '2003-1108',
 '2003-1110',
 '2003-1111',
 '2003-1112',
 '2003-1113',
 '2003-1114',
 '2003-1115',
 '2003-1116',
 '2003-1117',
 '2003-1119',
 '2003-1120',
 '2003-1122',
 '2003-1123',
 '2003-1124',
 '2003-1125',
 '2003-1126',
 '2003-1127',
 '2003-1128',
 '2003-1129',
 '2003-1130',
 '2003-1131',
 '2003-1132',
 '2003-1133',
 '2003-1135',
 '2003-1137',
 '2003-1138',
 '2003-1139',
 '2003-1140',
 '2003-1141',
 '2003-1142',
 '2003-1143',
 '2003-1144',
 '2003-1145',
 '2003-1147',
 '2003-1148',
 '2003-1149',
 '2003-1150',
 '2003-1151',
 '2003-1152',
 '2003-1153',
 '2003-1154',
 '2003-1155',
 '2003-1156',
 '2003-1157',
 '2003-1158',
 '2003-1159',
 '2003-1160',
 '2003-1161',
 '2003-1162',
 '2003-1163',
 '2003-1164',
 '2003-1165',
 '2003-1166',
 '2003-1168',
 '2003-1169',
 '2003-1170',
 '2003-1171',
 '2003-1172',
 '2003-1173',
 '2003-1174',
 '2003-1175',
 '2003-1176',
 '2003-1177',
 '2003-1178',
 '2003-1179',
 '2003

In [46]:
team_embeddings.shape

torch.Size([14311, 128])

In [47]:
target = target[['TeamAID','TeamBID','Win']]

In [48]:
team_embeddings[0]

tensor([-1.1377e-01, -2.8639e-01,  2.4650e-01,  8.5858e-01,  7.4697e-02,
        -5.3307e-02, -1.9253e+00,  1.3090e-01, -1.0276e-01,  3.9500e-01,
         4.1901e-01, -1.5478e-01, -1.1822e-01,  1.3809e+00, -3.3521e-01,
        -4.4488e-01, -4.2639e-01, -1.8628e-01,  1.5618e+00,  3.2168e-03,
         9.4336e-02, -4.1757e+00,  1.0627e-01, -1.5540e-01,  2.0620e-01,
        -4.6126e-02, -2.2289e-01, -2.4052e-02, -2.8870e-01, -9.6145e-02,
        -8.1023e-02, -2.5925e-02, -1.9697e-02, -1.6850e-01,  3.2345e-02,
        -6.6425e-02,  3.3214e-01, -6.0257e+00,  3.7433e-01,  1.1287e-01,
        -2.2679e-01, -5.0447e-01,  5.3854e-03,  6.4771e-01, -7.3023e-02,
         2.3965e-01,  8.6048e-02,  1.3067e-01, -3.0582e-01, -2.8880e-01,
        -6.3703e-01,  2.9371e-02,  1.8857e-01,  1.1952e-01,  4.6658e-01,
         1.8884e-01,  2.5700e-02, -1.1427e-01,  2.1326e-01,  7.5039e-02,
         1.9226e-01, -1.0921e+00,  4.2021e+00,  8.5746e-01,  3.6227e-01,
         4.7019e-04, -5.5777e-01,  3.0646e+00,  1.3

In [49]:
class MatchDatasetEmb(Dataset):
    def __init__(self, match_data, embedding_matrix):
        """
        match_data: list of (teamA_id, teamB_id, label)
        embedding_matrix: tensor (num_teams, d_model)
        """
        self.matches = match_data
        self.emb = embedding_matrix

    def __len__(self):
        return len(self.matches)

    def __getitem__(self, idx):
        row = self.matches.iloc[idx]
        teamA_id = row["TeamAID"]
        teamB_id = row["TeamBID"]
        label = row["Win"]

        # match team embeddings based on their keys
        teamA = team_ids.index(teamA_id)
        teamB = team_ids.index(teamB_id)

        zA = self.emb[teamA]
        zB = self.emb[teamB]

        return zA, zB, torch.tensor(label, dtype=torch.float32)

In [50]:
class MatchPredictorEmb(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, zA, zB):
        x = torch.cat([
            zA - zB,
            zA * zB,
            zA,
            zB
        ], dim=-1)

        return self.classifier(x)  # logits

In [ ]:
def train_classifier(model, dataloader, epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for zA, zB, y in dataloader:
            zA = zA.to(device)
            zB = zB.to(device)
            y = y.to(device)

            logits = model(zA, zB).squeeze()
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

In [30]:
from torch.utils.data import DataLoader

In [51]:
matchups = MatchDatasetEmb(target, team_embeddings)

matchupsdataloader = DataLoader(
    matchups,
    batch_size=64,
    shuffle=True
)

In [52]:
matchups[1]

(tensor([-2.7897e-01, -6.6485e-01, -3.4576e-02,  5.9416e-01,  3.7148e-01,
          1.3864e-01, -2.4226e+00, -1.7819e-01,  2.7734e-01, -3.9808e-02,
          6.9661e-01, -4.1086e-01, -7.4767e-02,  1.3136e+00, -5.7345e-01,
          7.2766e-02,  3.1706e-02,  2.8664e-01,  1.4845e+00, -6.6781e-02,
         -1.4799e-01, -3.9575e+00,  3.6084e-01, -8.8403e-02,  2.1637e-01,
         -2.6603e-01,  2.0518e-01, -3.3468e-02,  1.3487e-01, -7.0101e-02,
          1.4695e-01, -3.0083e-01, -6.1834e-02, -7.4166e-02, -1.5838e-01,
          2.5275e-01,  3.7482e-01, -5.6887e+00, -9.8245e-02, -5.5956e-02,
          1.1529e-01, -2.7704e-01,  1.7861e-01,  5.4600e-01, -1.2522e-01,
         -1.3231e-01, -3.1031e-02,  2.5281e-01, -1.2013e-01,  2.7238e-02,
         -8.9716e-01, -4.6938e-03, -8.4604e-02,  6.2345e-02, -2.9658e-01,
         -1.1372e-01, -4.7444e-01, -1.2512e-01, -1.4758e-01, -1.0260e-01,
         -1.3178e-01, -6.1667e-01,  4.4299e+00,  7.6576e-01,  2.1321e-01,
         -4.9387e-02, -5.0536e-01,  3.

In [ ]:
d_model = team_embeddings.shape[1]

classifier = MatchPredictorEmb(d_model)

train_classifier(classifier, matchupsdataloader, epochs=10, lr=1e-3)

Epoch 1/10 | Loss: 0.5002
Epoch 2/10 | Loss: 0.4971
Epoch 3/10 | Loss: 0.4963
Epoch 4/10 | Loss: 0.4955
Epoch 5/10 | Loss: 0.4950
Epoch 6/10 | Loss: 0.4946
Epoch 7/10 | Loss: 0.4943
Epoch 8/10 | Loss: 0.4941
Epoch 9/10 | Loss: 0.4940
Epoch 10/10 | Loss: 0.4938


In [35]:
def predict(model, teamA_id, teamB_id, embedding_matrix):
    device = next(model.parameters()).device

    teamA = team_ids.index(teamA_id)
    teamB = team_ids.index(teamB_id)

    
    zA = embedding_matrix[teamA].unsqueeze(0).to(device)
    zB = embedding_matrix[teamB].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(zA, zB)
        prob = torch.sigmoid(logits)

    return prob.item()

In [ ]:
p = predict(classifier, teamA_id='2023-1104', teamB_id='2023-1328', embedding_matrix=team_embeddings)
print("P(A wins):", p)

P(A wins): 0.9383870959281921


### Combining Training Encoder and Model

In [53]:
import torch
import torch.nn as nn

class FullModel(nn.Module):
    def __init__(self, encoder, d_model):
        super().__init__()
        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, teamA, teamB):
        reconA, zA = self.encoder(teamA)
        reconB, zB = self.encoder(teamB)

        x = torch.cat([
            zA - zB,
            zA * zB,
            zA,
            zB
        ], dim=-1)

        logits = self.classifier(x)

        return reconA, reconB, logits

In [54]:
def train_joint(model, dataloader, epochs=10, lr=1e-3, lambda_match=1.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    recon_loss_fn = nn.MSELoss()
    match_loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for teamA, teamB, y in dataloader:
            teamA = teamA.to(device)
            teamB = teamB.to(device)
            y = y.to(device)

            reconA, reconB, logits = model(teamA, teamB)

            # Reconstruction loss
            loss_recon = recon_loss_fn(reconA, teamA) + recon_loss_fn(reconB, teamB)

            # Match prediction loss
            loss_match = match_loss_fn(logits.squeeze(), y)

            # Combined loss
            loss = loss_recon + lambda_match * loss_match

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(dataloader):.4f}")

In [55]:
fullmodel = FullModel(encoder,d_model)

In [56]:
train_joint(fullmodel,matchupsdataloader)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x128 and 22x128)